# Clean-protocol test: volpath (and volpath + cyclic)

What is tested: the two winners of the feature-screening screen
(`feature_screening.ipynb` section 4) under the clean adoption protocol —
the unchanged baseline pipeline (per-generator XGBoost, train 2015-2021,
train_tail early stopping, seed 0, lr 0.05, 400 trees, val 2022) with:

1. **baseline**: the 193 features (reproduction check, micro AUC 0.9802).
2. **+volpath**: 193 + the reservoir volume trajectory through the case
    week — per-reservoir volume fraction at each of the 7 day boundaries
    plus per-generator local capacity-weighted fractions per day
    (screen delta was +0.0017).
3. **+volpath+cyclic**: additionally the cyclic hour-of-day and
    day-of-week sin/cos encodings (screen deltas +0.0017 and +0.0008).

Adoption bar: beat 0.9802 beyond the measured noise floor (~0.001).
Printed: micro AUC per configuration, per-generator AUCs with deltas
vs the baseline, and tree counts. Expected runtime ~30-40 minutes.

In [1]:
import gc

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168

In [2]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                 if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("cases:", len(target), " generators:", len(GENERATORS))

cases: 2922  generators: 14


## Feature engineering and split

What is tested: the unchanged 193-column feature set and the baseline
split (train starts < 2022-01-01, stop on the last 90 days of train,
val = full 2022), copied verbatim from
`notebooks/xgboost_baseline.ipynb`.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


split = pd.Timestamp("2022-01-01", tz="UTC")
train_cases = target[target["starttime"] < split].reset_index(drop=True)
val_cases = target[target["starttime"] >= split].reset_index(drop=True)
stop_cases = train_cases[
    train_cases["starttime"] >= split - pd.Timedelta(days=90)
].reset_index(drop=True)
print("train cases:", len(train_cases), " stop cases:", len(stop_cases),
      " val cases:", len(val_cases))

Xtr_base = feature_matrix(train_cases)
Xst_base = feature_matrix(stop_cases)
Xva_base = feature_matrix(val_cases)
print("feature matrix:", Xtr_base.shape)

train cases: 2557  stop cases: 90  val cases: 365
feature matrix: (429576, 186)


## Extra features: volpath and cyclic

What is tested: the volpath group — per-reservoir volume fraction at
each of the 7 day boundaries of the case week (value constant within
each day) — and the cyclic group (hour-of-day and day-of-week sin/cos),
built for the train, stop, and val cases. Printed: shapes and a NaN
check (a misaligned volume lookup would surface as NaNs here).

In [4]:
def extra_features(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    E = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        E[name] = v

    def add_day(name, v, d):
        week = np.zeros((n, 7), dtype=np.float32)
        week[:, d - 1] = v
        E[name] = np.repeat(week, 24, axis=1).reshape(-1)

    for d in range(1, 8):
        vols = lookup(vol_wide, starts + pd.Timedelta(days=d), RESERVOIRS)
        for j, res in enumerate(RESERVOIRS):
            add_day(f"volfrac_d{d}__{res}", vols[:, j] / CAPACITY[res], d)

    hod = ts.hour.to_numpy()
    dow = ts.dayofweek.to_numpy()
    add("hod_sin", np.sin(2 * np.pi * hod / 24))
    add("hod_cos", np.cos(2 * np.pi * hod / 24))
    add("dow_sin", np.sin(2 * np.pi * dow / 7))
    add("dow_cos", np.cos(2 * np.pi * dow / 7))

    return pd.DataFrame(E, dtype=np.float32)


EXTRA_TRAIN = extra_features(train_cases)
EXTRA_STOP = extra_features(stop_cases)
EXTRA_VAL = extra_features(val_cases)
print("extra matrices:", EXTRA_TRAIN.shape, EXTRA_STOP.shape, EXTRA_VAL.shape)
print("NaNs in extras:", int(EXTRA_TRAIN.isna().to_numpy().sum()),
      int(EXTRA_STOP.isna().to_numpy().sum()),
      int(EXTRA_VAL.isna().to_numpy().sum()))

extra matrices: (429576, 123) (15120, 123) (61320, 123)
NaNs in extras: 0 0 0


## Fit: baseline vs +volpath vs +volpath+cyclic

What is tested: the three configurations under the identical clean
protocol. Printed: micro AUC per configuration and a per-generator
table of AUCs with deltas vs the baseline. Comparison target: baseline
micro AUC 0.9802; adoption requires beating it beyond the ~0.001
noise floor.

In [5]:
def volpath_cols(E, gen):
    feeders = FEEDERS[gen]
    cap = sum(CAPACITY[r] for r in feeders)
    X = E[[f"volfrac_d{d}__{r}" for d in range(1, 8) for r in RESERVOIRS]].copy()
    for d in range(1, 8):
        X[f"local_volfrac_d{d}"] = sum(
            E[f"volfrac_d{d}__{r}"] * CAPACITY[r] for r in feeders) / cap
    return X


def volpath_cyclic_cols(E, gen):
    return pd.concat([volpath_cols(E, gen),
                      E[["hod_sin", "hod_cos", "dow_sin", "dow_cos"]]], axis=1)


def fit_all(cols_fn=None):
    preds, rows = {}, []
    for gen in GENERATORS:
        Xtr = generator_features(Xtr_base, gen)
        Xst = generator_features(Xst_base, gen)
        Xva = generator_features(Xva_base, gen)
        if cols_fn is not None:
            Xtr = pd.concat([Xtr, cols_fn(EXTRA_TRAIN, gen)], axis=1)
            Xst = pd.concat([Xst, cols_fn(EXTRA_STOP, gen)], axis=1)
            Xva = pd.concat([Xva, cols_fn(EXTRA_VAL, gen)], axis=1)
        ytr = labels_for(train_cases, gen)
        yst = labels_for(stop_cases, gen)
        yva = labels_for(val_cases, gen)

        if np.unique(ytr).size == 1:
            preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
            rows.append({"generator": gen, "trees": 0, "AUC": None})
        else:
            clf = xgb.XGBClassifier(
                n_estimators=400, learning_rate=0.05, max_depth=7,
                subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                eval_metric="auc", early_stopping_rounds=25, random_state=0,
            )
            clf.fit(Xtr, ytr, eval_set=[(Xst, yst)], verbose=False)
            preds[gen] = clf.predict_proba(Xva)[:, 1].astype(np.float32)
            rows.append({"generator": gen, "trees": clf.best_iteration + 1,
                         "AUC": roc_auc_score(yva, preds[gen])})
            del clf
        del Xtr, Xst, Xva
        gc.collect()

    y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
    p_all = np.concatenate([preds[g] for g in GENERATORS])
    return pd.DataFrame(rows).set_index("generator"), roc_auc_score(y_all, p_all)


CONFIGS = {
    "baseline": None,
    "volpath": volpath_cols,
    "volpath+cyclic": volpath_cyclic_cols,
}
tables, micros = {}, {}
for name, fn in CONFIGS.items():
    tables[name], micros[name] = fit_all(fn)
    print(f"{name:16s} micro AUC {micros[name]:.4f}")

cmp = pd.DataFrame({name: tables[name]["AUC"] for name in CONFIGS})
cmp["d_volpath"] = cmp["volpath"] - cmp["baseline"]
cmp["d_volpath+cyclic"] = cmp["volpath+cyclic"] - cmp["baseline"]
cmp.round(4)

baseline         micro AUC 0.9802
volpath          micro AUC 0.9804
volpath+cyclic   micro AUC 0.9806


,baseline,volpath,volpath+cyclic,d_volpath,d_volpath+cyclic
generator,,,,,
Byrte_G1,0.9951,0.9953,0.9954,0.0002,0.0004
Haukeli_G1,0.9352,0.9400,0.9335,0.0048,-0.0017
Hogga_G1,0.9722,0.9729,0.9729,0.0006,0.0006
Kjela_G1,0.9789,0.9805,0.9823,0.0016,0.0034
Lio_G1,0.9225,0.9144,0.9195,-0.0081,-0.0030
Songa_G1,0.9966,0.9951,0.9960,-0.0015,-0.0005
Tokke_G1,0.9928,0.9927,0.9928,-0.0001,0.0001
Tokke_G2,0.9839,0.9833,0.9831,-0.0006,-0.0007
Tokke_G3,0.9886,0.9883,0.9879,-0.0003,-0.0007
